# Elastic 2-D topography benchmark — September 2026

Progressive validation of OPT3 and OPT5 against SPECFEM2D. The flat homogeneous free surface is validated over a longer physical window before introducing Kirishima topography (Stage 1). Stage 1's own principal figures still deliberately exclude FD3 (see the Stage 1 heading below). The Kirishima heterogeneous-model stages that follow (2a: flat free surface, 2b: real topography) compare all four solvers — FD3, OPT3, OPT5 and SPECFEM2D.

In [ ]:
import Pkg
function find_flexopt_root(start=pwd())
    directory = abspath(start)
    while true
        isfile(joinpath(directory, "src", "flexOPT.jl")) && return directory
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    haskey(ENV, "FLEXOPT_ROOT") && return abspath(ENV["FLEXOPT_ROOT"])
    error("Cannot locate flexOPT; set ENV[\"FLEXOPT_ROOT\"]")
end
flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
using CairoMakie, JLD2, LinearAlgebra, Statistics
CairoMakie.activate!(type="png")
@show VERSION Threads.nthreads() Base.active_project()


## Stage 1 — long flat free-surface benchmark

The three nominal worker spacings 1000, 750 and 500 m become physical spacings 500, 375 and 250 m because the long profile activates the established factor-two refinement. Duration is 14 s and the half-domain is 90 km.

In [ ]:
runLongFlatBenchmark = false # expensive: set true explicitly
longBenchmarkCommand = `$(Base.julia_cmd()) --project=$(flexopt_root)
    --startup-file=no --threads=8
    $(joinpath(flexopt_root, "scripts", "run_flat_free_surface_convergence.jl"))
    --long-opt5 1000 750 500`
@show longBenchmarkCommand
runLongFlatBenchmark && run(longBenchmarkCommand)


In [ ]:
longDataDirectory = joinpath(flexopt_root, "data",
    "elastic2d_convergence", "flat_free_surface_long_opt5")
longFiles = isdir(longDataDirectory) ? sort(filter(
    file -> endswith(file, ".jld2"), readdir(longDataDirectory; join=true))) : String[]
longResults = sort([load(file, "result") for file in longFiles];
    by=result -> result.spacing_m, rev=true)
isempty(longResults) && @warn "No long OPT5 products yet; enable runLongFlatBenchmark once"
!isempty(longResults) && display([(dx=result.spacing_m,
    duration=result.duration_s, dt_OPT3=result.OPT3.dt,
    dt_OPT5=result.OPT5.dt, dt_SPECFEM=result.SPECFEM2D.dt)
    for result in longResults])


In [ ]:
function compact_trace(result, method, component, station)
    block = getproperty(result, method)
    values = getproperty(block, component)
    method === :SPECFEM2D ? values[station] :
        (time=values.time, values=values.values[:, station])
end
if !isempty(longResults)
    result = last(longResults)
    methods = (:OPT3, :OPT5, :SPECFEM2D)
    colors = Dict(:OPT3 => :darkorange, :OPT5 => :purple,
        :SPECFEM2D => :black)
    lateFigure = Figure(size=(1200, 225 * length(result.receivers)))
    for (station, receiver) in pairs(result.receivers)
        axis = Axis(lateFigure[station, 1]; xlabel="time (s)",
            ylabel="u_z (m)",
            title="late absolute phases, station $station: x=$(receiver.x/1e3), z=$(receiver.z/1e3) km",
            limits=(0.0, result.duration_s, nothing, nothing))
        for method in methods
            trace = compact_trace(result, method, :z, station)
            lines!(axis, trace.time, trace.values; color=colors[method],
                label=String(method))
        end
        station == 1 && axislegend(axis; position=:rt)
    end
    display(lateFigure)
end


In [ ]:
function sample_trace(trace, times)
    [begin
        upper = searchsortedfirst(trace.time, time)
        if upper <= 1
            trace.values[1]
        elseif upper > length(trace.time)
            trace.values[end]
        else
            fraction = (time-trace.time[upper-1]) /
                (trace.time[upper]-trace.time[upper-1])
            trace.values[upper-1] + fraction *
                (trace.values[upper]-trace.values[upper-1])
        end
    end for time in times]
end
if length(longResults) >= 2
    finest = last(longResults)
    convergenceFigure = Figure(size=(1100, 500))
    axis = Axis(convergenceFigure[1, 1]; xscale=log10, yscale=log10,
        xlabel="spacing (m)", ylabel="global relative RMS to finest",
        title="14 s flat-surface convergence")
    for (method, color) in ((:OPT3, :darkorange), (:OPT5, :purple),
            (:SPECFEM2D, :black))
        errors = Float64[]
        spacings = Float64[]
        for candidate in longResults[1:end-1]
            referenceValues, candidateValues = Float64[], Float64[]
            for station in eachindex(finest.receivers)
                reference = compact_trace(finest, method, :z, station)
                trial = compact_trace(candidate, method, :z, station)
                times = range(max(first(reference.time), first(trial.time)),
                    min(last(reference.time), last(trial.time)); length=1801)
                append!(referenceValues, sample_trace(reference, times))
                append!(candidateValues, sample_trace(trial, times))
            end
            push!(errors, norm(candidateValues-referenceValues) /
                norm(referenceValues))
            push!(spacings, candidate.spacing_m)
        end
        scatterlines!(axis, spacings, errors; color, label=String(method))
    end
    axislegend(axis)
    display(convergenceFigure)
end


## Stage 2b — Kirishima non-flat topography

This stage remains disabled until OPT3 and OPT5 complete the 14 s flat run without early stopping and show consistent convergence against SPECFEM2D (Stage 1 above). As of now only the dx=500 m point of that long run exists; dx=750 m and dx=1000 m are still missing, so this stage is being built ahead of that gate clearing, at the user's explicit request. Stage 2a above runs the same four solvers on the same heterogeneous NIED model with a flattened free surface, to isolate topography's own effect before this stage's real relief.

`topography-bootstrap`/`topography-model` build the Kirishima cross-section itself — real topography and the NIED heterogeneous velocity model — reusing only `constructLocalBox`/`getParamsAndTopo` from `SimuKirishima.ipynb`, at the same fine Δx=Δz=100 m used there (coarser grids do not converge; see Stage 1). "Air" here is not modelled as a medium: `material2D` only marks where the traction-free surface sits, exactly like `applyFreeSurface`/`material` in `HomogeneousElastic2DBenchmark_FreeSurface.ipynb`.

`topography-opt-*` and `topography-specfem` propagate FD3, OPT3, OPT5 and SPECFEM2D on that model and are **not** SimuKirishima's own OPT3/FD3 cells — those run the OPT operator at a coarsened 200 m stride and use a simplified surface closure that does not match what `KirishimaElastic2DBenchmark.ipynb` documents. OPT3 and OPT5 instead reuse the audited `clipped_available` weak-form closure from `HomogeneousElastic2DBenchmark_FreeSurface.ipynb` (`elastic2D_OPT3`/`elastic2D_OPT3_surface_clipped_available` and their OPT5 counterparts, cache-compatible with that notebook since the recipe only depends on stencil geometry, not on this model's fields), adapted to `material2D`'s real, non-flat surface normals. FD3 uses the explicit `elasticWave2D` reference solver (same module as `HomogeneousElastic2DBenchmark_FreeSurface.ipynb`'s `fd-propagation` cell), included via `topography-opt-bootstrap`.

FD3, OPT3, OPT5 and the SPECFEM2D run are all expensive on this 80 km × 42 km, 100 m grid and default to `true` in `topography-opt-config` (`runKirishimaFD3`, `runKirishimaOPT3`, `runKirishimaOPT5`, `runKirishimaSPECFEM2D`); set any to `false` to skip that solver. `topography-comparison` and `topography-video` degrade gracefully (a `@warn` and nothing else) until both runs exist.

In [ ]:
# Only the GeoPoints/planet1D module chain, needed to build the real
# Kirishima cross-section below. No Metal/backend and no flexOPT
# include here: those are only needed once the propagation part of
# this stage is wired up (see the note above).
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints

In [ ]:
# Kirishima cross-section: real topography + NIED heterogeneous
# velocity model. Reuses only SimuKirishima.ipynb's construction
# primitives (constructLocalBox, getParamsAndTopo) at its own fine
# Δx=Δz=100 m — not its 3-D model (unneeded for a 2-D benchmark)
# and not its propagation cells.
kirishimaSummit = GeoPoint(dmsToDecimal(31, 56, 03), dmsToDecimal(130, 51, 42))
p0 = kirishimaSummit
Δx2D = 100.0 # m
Δz2D = 100.0 # m
altMax = 2.e3 # m
altMin = -40.e3 # m
horizontalHalfWidth = 40.e3 # m; the model spans 80 km horizontally

boxGrids2D = constructLocalBox(
    p0, Δx2D, Δz2D,
    -horizontalHalfWidth, horizontalHalfWidth,
    altMin, altMax,
)

niedVelocitySource = DEFAULT_NIED_VELOCITY_SOURCE[]
velocitySourceKey = nied_velocity_source_key(niedVelocitySource)
gridCacheTag2D = "$(join(size(boxGrids2D.allGridsInGeoPoints), 'x'))_" *
    "d$(boxGrids2D.Δx)_$(boxGrids2D.Δz)"
modelCacheName2D = "seismicModel2D_Kirishima_$(gridCacheTag2D)_NIED_$(velocitySourceKey)"
seismicModel2D = lazyProduceOrLoad(
    modelCacheName2D,
    getParamsAndTopo,
    boxGrids2D.allGridsInGeoPoints,
    boxGrids2D.effectiveRadii,
    0.1;
    velocity_model=:NIED,
    nied_source=niedVelocitySource,
    nied_confidence_max=0.8,
    nied_outside=:planet1D,
    nied_low_confidence=:planet1D,
)
@assert size(seismicModel2D.ρ) == size(boxGrids2D.allGridsInGeoPoints)

# getParamsAndTopo uses ρAir=0.001 by default; 0.01 cleanly separates
# rock from air/void (same threshold as SimuKirishima.ipynb).
air_density_cutoff = 0.01
material2D = seismicModel2D.ρ .> air_density_cutoff

xCoordinates2D = [p.xz[1] for p in boxGrids2D.allGridsInCartesian[:, 1]]
zCoordinates2D = [p.xz[2] for p in boxGrids2D.allGridsInCartesian[1, :]]
topographicSurfaceIndices2D = [
    findlast(@view material2D[ix, :]) for ix in axes(material2D, 1)]
@assert all(!isnothing, topographicSurfaceIndices2D) "a column is all air/void: widen altMax, or all solid: widen altMin"
topographicSurfaceIndices2D = Int.(topographicSurfaceIndices2D)
surfaceZ2D = [zCoordinates2D[k] for k in topographicSurfaceIndices2D]

@show size(material2D) count(material2D) extrema(surfaceZ2D)
@show seismicModel2D.velocity_model seismicModel2D.nied_source
@show extrema(seismicModel2D.Vpv[material2D]) extrema(seismicModel2D.Vsv[material2D])

topographyFigure = Figure(size=(900, 380))
topographyAxis = Axis(topographyFigure[1, 1];
    xlabel="x (km)", ylabel="z (km)", aspect=DataAspect(),
    title="Kirishima cross-section: topography-following material mask")
vsRange = extrema(seismicModel2D.Vsv[material2D])
heatmap!(topographyAxis, xCoordinates2D ./ 1e3, zCoordinates2D ./ 1e3,
    seismicModel2D.Vsv; colormap=:viridis, colorrange=vsRange)
lines!(topographyAxis, xCoordinates2D ./ 1e3, surfaceZ2D ./ 1e3;
    color=:white, linewidth=2, label="material/air interface")
Colorbar(topographyFigure[1, 2]; colormap=:viridis, colorrange=vsRange,
    label="Vs (km/s)")
axislegend(topographyAxis; position=:rb)
display(topographyFigure)

In [ ]:
# Metal/backend, flexOPT and specfemBenchmark — needed from here on for
# the OPT3/SPECFEM2D propagation. Mirrors the proven bootstrap in
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb, not SimuKirishima.ipynb's
# (which hard-errors when Metal is unavailable; detect_backend() already
# falls back to CPU on its own).
using Metal
include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
# Ordinary explicit FD operators, needed for FD3 in Stage 2a/2b below.
# Not included by topography-bootstrap above (that cell only loads the
# GeoPoints/planet1D chain used to build the cross-section).
include(joinpath(flexopt_root, "src", "elasticWave2D.jl"))
backend isa KernelAbstractions.CPU &&
    @warn("Running the OPT3 recipe construction on CPU; this will be slow")
include(joinpath(flexopt_root, "src", "flexOPT.jl"))
include(joinpath(flexopt_root, "src", "specfemBenchmark.jl"))
using .elasticWave2D, .flexOPT, .specfemBenchmark
set_default_form!(:weak)
@assert DEFAULT_FORM[] === :weak
@show backend

In [ ]:
runKirishimaFD3 = true # expensive: set true explicitly
runKirishimaOPT3 = true # expensive: set true explicitly
runKirishimaOPT5 = true # expensive: set true explicitly
runKirishimaSPECFEM2D = true # expensive: set true explicitly

dxOPT = Δx2D
@assert Δx2D == Δz2D "the OPT3 recipe below assumes a square grid"
vpFieldOPT = seismicModel2D.Vpv .* 1e3   # m/s
vsFieldOPT = seismicModel2D.Vsv .* 1e3   # m/s
rhoFieldOPT = seismicModel2D.ρ .* 1e3    # kg/m^3
vpMaximum = maximum(vpFieldOPT[material2D])
# Conditioning constant only, exactly like rho0 in
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb: ρ, μ and λ are all
# divided by it, so it cancels out of the physical PDE it discretises.
rhoReference = median(rhoFieldOPT[material2D])
temporalRefinement = 2
dtOPT = 0.20 * dxOPT / (sqrt(2) * vpMaximum * temporalRefinement)

optFormulation = :scaled_coordinates
recipeSpacing = (1.0, 1.0, 1.0)
muOPT = (rhoFieldOPT .* vsFieldOPT.^2 ./ rhoReference) .* (dtOPT/dxOPT)^2
lambdaOPT = (rhoFieldOPT .* (vpFieldOPT.^2 .- 2vsFieldOPT.^2) ./ rhoReference) .*
    (dtOPT/dxOPT)^2
rhoOPT = rhoFieldOPT ./ rhoReference
muOPT[.!material2D] .= 0.0
lambdaOPT[.!material2D] .= 0.0

epicentreX2D = -10e3 # same epicentre as SimuKirishima.ipynb
sourceDepthBelowTopography2D = 2e3
sourceIX2D = argmin(abs.(xCoordinates2D .- epicentreX2D))
surfaceAltitudeAtSource2D = surfaceZ2D[sourceIX2D]
sourcePhysicalZ2D = surfaceAltitudeAtSource2D - sourceDepthBelowTopography2D
sourceIZ2D = argmin(abs.(zCoordinates2D .- sourcePhysicalZ2D))
@assert material2D[sourceIX2D, sourceIZ2D] "the source falls outside solid material; increase sourceDepthBelowTopography2D"

simulationDuration2D = 30.0 # s; includes several boundary reflections
outputSampling2D = 0.10 # s between stored/video frames
sourceFrequency2D = median(vsFieldOPT[material2D]) / (10dxOPT)
sourceDelay2D = min(12dtOPT, 0.25simulationDuration2D)
sourceForceAmplitude2D = 1.0e10 # N/m: line force in a unit-thickness slice
rickerSource2D(time) = begin
    a = π * sourceFrequency2D * (time - sourceDelay2D)
    (1 - 2a^2) * exp(-a^2)
end
# Soft check only, not a hard @assert: Vs dips locally near the surface of
# a volcanic edifice, and a single low-velocity cell should not block the
# whole run the way it correctly does in the homogeneous benchmark.
pointsPerSWavelength2D = minimum(vsFieldOPT[material2D]) / (sourceFrequency2D * dxOPT)

# Five receivers along the topographic surface, sampled two grid cells
# below the local elevation: the boundary row itself mixes surface/void
# degrees of freedom by construction and is not a clean displacement probe.
receiverOffsetsX2D = [-30e3, -20e3, 0.0, 20e3, 30e3]
receiverDepthBelowSurface2D = 2dxOPT
receiverGrid2D = map(receiverOffsetsX2D) do xr
    ix = argmin(abs.(xCoordinates2D .- xr))
    (x=xCoordinates2D[ix], z=surfaceZ2D[ix] - receiverDepthBelowSurface2D)
end
@assert all(receiverGrid2D) do r
    material2D[argmin(abs.(xCoordinates2D .- r.x)), argmin(abs.(zCoordinates2D .- r.z))]
end "a receiver falls outside solid material"

dtSPECFEM2D = dtOPT # exact common physical time step for OPT3 and SPECFEM2D
@show dxOPT dtOPT vpMaximum rhoReference
@show sourceFrequency2D sourceDelay2D pointsPerSWavelength2D
@show receiverGrid2D

## Stage 2a -- Kirishima heterogeneous model, flat free surface

Before comparing against the real (rough) Kirishima topography below,
this stage isolates the effect of the free-surface *shape* from the
effect of the heterogeneous NIED velocity structure: it keeps
`seismicModel2D`'s real heterogeneous Vp/Vs/rho unchanged, but flattens
the free surface to `flatReferenceZ2D = minimum(surfaceZ2D)`, the
lowest point of the real topography. Because that is the lowest point
anywhere along the cross-section, every solid cell in the flattened
mask (`materialFlat2D`) was already solid in the real model -- no
material is invented, only the topography above `flatReferenceZ2D` is
discarded.

Four solvers are compared, all at the same `dxOPT`/`dtOPT` grid used
by the rough stage below: FD3 (the explicit `elasticWave2D` reference,
via `prepare_elastic_wave_2d`), OPT3 and OPT5 (both using the
`clipped_available` weak free-surface closure validated in
`HomogeneousElastic2DBenchmark_FreeSurface.ipynb`, including the OPT5
generalisation fixed there), and SPECFEM2D as the reference. The
comparison figure plots absolute (non-normalized) displacement, per
request -- no normalized-amplitude plot is produced here.

Run flags `runFlatKirishimaFD3`/`OPT3`/`OPT5`/`SPECFEM2D` in the next
cell default to `true`; set any to `false` to skip that solver.

In [ ]:
# Stage 2a config: run flags + flattened material mask + flat source/receiver
# geometry. Reuses seismicModel2D's real heterogeneous NIED velocity/density
# field unchanged -- only the free-surface geometry is flattened, at the
# lowest point of the real topography (flatReferenceZ2D), which guarantees
# every solid cell below it was already solid in the real Kirishima model:
# flattening never invents material.
runFlatKirishimaFD3 = true # expensive: set true explicitly
runFlatKirishimaOPT3 = true # expensive: set true explicitly
runFlatKirishimaOPT5 = true # expensive: set true explicitly
runFlatKirishimaSPECFEM2D = true # expensive: set true explicitly

flatReferenceZ2D = minimum(surfaceZ2D)
materialFlat2D = material2D .& reshape(zCoordinates2D .<= flatReferenceZ2D, 1, :)
@assert !any(materialFlat2D .& .!material2D) "flattening must never add material absent from the real topography"
surfaceZFlat2D = fill(flatReferenceZ2D, length(xCoordinates2D))

muOPTFlat2D = muOPT .* materialFlat2D
lambdaOPTFlat2D = lambdaOPT .* materialFlat2D

sourcePhysicalZFlat2D = flatReferenceZ2D - sourceDepthBelowTopography2D
sourceIZFlat2D = argmin(abs.(zCoordinates2D .- sourcePhysicalZFlat2D))
@assert materialFlat2D[sourceIX2D, sourceIZFlat2D] "the flat-stage source falls outside solid material; increase sourceDepthBelowTopography2D"

receiverGridFlat2D = map(receiverOffsetsX2D) do xr
    ix = argmin(abs.(xCoordinates2D .- xr))
    (x=xCoordinates2D[ix], z=flatReferenceZ2D - receiverDepthBelowSurface2D)
end
@assert all(receiverGridFlat2D) do r
    materialFlat2D[argmin(abs.(xCoordinates2D .- r.x)), argmin(abs.(zCoordinates2D .- r.z))]
end "a flat-stage receiver falls outside solid material"

@show flatReferenceZ2D count(materialFlat2D) count(material2D)
@show sourcePhysicalZFlat2D sourceIZFlat2D
@show receiverGridFlat2D


In [ ]:
# FD3: ordinary explicit finite-difference reference, same elasticWave2D
# module and call pattern as HomogeneousElastic2DBenchmark_FreeSurface.ipynb's
# fd-propagation cell, but driven by this stage's real heterogeneous
# Vp/Vs/rho (seismicModel2D) restricted to the flattened mask.
cerjanFlat2D = CerjanBoundarySpec((24, 24), (24, 0); damping=0.0053)

preparedFD3Flat2D = if runFlatKirishimaFD3
    fdConfigFlat2D = ElasticThreePointConfig2D(
        pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, cfl=0.38,
    )
    bcFDFlat2D = boundary_geometry(materialFlat2D, (dxOPT, dxOPT); cerjan=cerjanFlat2D)
    modelFDFlat2D = (ρ=seismicModel2D.ρ, Vpv=seismicModel2D.Vpv, Vsv=seismicModel2D.Vsv)
    prepare_elastic_wave_2d(
        modelFDFlat2D, (dxOPT, dxOPT);
        material_mask=materialFlat2D,
        boundary_conditions=bcFDFlat2D,
        config=fdConfigFlat2D,
    )
else
    nothing
end
runFlatKirishimaFD3 && @show preparedFD3Flat2D.dt

# OPT3/OPT5 weak recipes: identical stencil-geometry parameters to
# topography-opt-operator below (and to
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb's own OPT3/OPT5 recipes),
# so cachedOPTRecipe2D's disk cache is shared across all three notebooks --
# only the material fields differ here.
optParameters2D = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
    "Δ" => recipeSpacing,
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    "variationalForm" => DEFAULT_FORM[],
    "taylor_inverse_mode" => :weak_operator_optimized,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "recipe_backend" => backend,
)
clippedSurfaceParameters2D = copy(optParameters2D)
clippedSurfaceParameters2D["pointsInSpace"] = (3, 2)
clippedSurfaceParameters2D["nuCentre"] = (2, 2)
clippedSurfaceParameters2D["fieldItpl"] = (
    ptsSpace=(3, 2), ptsTime=1, offsetSpace=(0.0, 0.0),
    offsetTime=1, YorderBspace=1, YorderBtime=-1,
)
clippedSurfaceParameters2D["trialFunctionRefPoints"] = (
    (1, 2, 3), (1, 2, 3), (1, 2, 3),
)
clippedSurfaceParameters2D["testIntegrationBounds"] = (
    (1.0, 3.0), (1.0, 2.0), (1.0, 3.0),
)
clippedSurfaceParameters2D["exactTaylorTotalDegree"] = 2

# Five-point OPT5 recipe and its clipped_available free-surface closure --
# the same 5->3 generalisation validated in
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb's opt-recipes/opt-operator
# cells (offsetSpace=2.0, centre=3, drop the two above-centre indices 4,5
# near the free surface).
opt5Parameters2D = copy(optParameters2D)
opt5Parameters2D["orderBspace"] = 1
opt5Parameters2D["pointsInSpace"] = 5
opt5Parameters2D["fieldItpl"] = merge(optParameters2D["fieldItpl"], (offsetSpace=2.0,))
opt5Parameters2D["materItpl"] = merge(optParameters2D["materItpl"], (offsetSpace=2.0,))
clippedSurfaceParametersOPT5_2D = copy(opt5Parameters2D)
clippedSurfaceParametersOPT5_2D["pointsInSpace"] = (5, 3)
clippedSurfaceParametersOPT5_2D["nuCentre"] = (3, 3)
clippedSurfaceParametersOPT5_2D["fieldItpl"] = (
    ptsSpace=(5, 3), ptsTime=1, offsetSpace=(0.0, 0.0),
    offsetTime=1, YorderBspace=1, YorderBtime=-1,
)
clippedSurfaceParametersOPT5_2D["trialFunctionRefPoints"] = (
    (1, 2, 3, 4, 5), (1, 2, 3, 4, 5), (1, 2, 3),
)
clippedSurfaceParametersOPT5_2D["testIntegrationBounds"] = (
    (1.0, 5.0), (1.0, 3.0), (1.0, 3.0),
)
clippedSurfaceParametersOPT5_2D["exactTaylorTotalDegree"] = 4

recipeCacheVersion2D = 9
function cachedOPTRecipe2D(parameters, prefix)
    cacheParameters = Dict{String,Any}(
        key => value for (key, value) in parameters if key != "recipe_backend")
    cacheParameters["recipe_cache_version"] = recipeCacheVersion2D
    function produceRecipe(config)
        runtimeParameters = Dict{String,Any}(config)
        pop!(runtimeParameters, "hash_id", nothing)
        pop!(runtimeParameters, "recipe_cache_version", nothing)
        runtimeParameters["recipe_backend"] = backend
        makeOPTsemiSymbolic(runtimeParameters)
    end
    myProduceOrLoad(produceRecipe, cacheParameters, "semiSymbolic", prefix)
end
optRecipe2D = cachedOPTRecipe2D(optParameters2D, "elastic2D_OPT3")
clippedSurfaceRecipe2D = cachedOPTRecipe2D(
    clippedSurfaceParameters2D, "elastic2D_OPT3_surface_clipped_available")
if runFlatKirishimaOPT5
    coefficientGatePath2D = joinpath(flexopt_root, "data",
        "elastic_lhs_coefficient_gate.jld2")
    isfile(coefficientGatePath2D) || error(
        "Run ElasticLHSOperatorAudit.ipynb before OPT5 propagation")
    coefficientGate2D = load(coefficientGatePath2D)
    @assert all(result.passed for result in values(coefficientGate2D["coefficient_health"]))
end
opt5Recipe2D = runFlatKirishimaOPT5 ?
    cachedOPTRecipe2D(opt5Parameters2D, "elastic2D_OPT5") : nothing
clippedSurfaceRecipeOPT5_2D = runFlatKirishimaOPT5 ?
    cachedOPTRecipe2D(clippedSurfaceParametersOPT5_2D,
        "elastic2D_OPT5_surface_clipped_available") : nothing

function prepare_surface_geometry_flat2D(recipe, name, mask; points_in_space=3,
        models=[rhoOPT, lambdaOPTFlat2D, muOPTFlat2D])
    surfacePoints = getModelPoints(models[1], points_in_space,
        recipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    surfaceFamily = (models=models, modelPoints=surfacePoints,
        Δ=recipeSpacing, modelName=name)
    numerical = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => recipe, "modelFam" => surfaceFamily,
        "absorbingBoundaries" => cerjan_padding(cerjanFlat2D),
        "maskedRegionInSpace" => mask,
        "boundaryConditions" => nothing,
        "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numerical)
end

bcOPTFlat2D = boundary_geometry(materialFlat2D, (dxOPT, dxOPT);
    free_surface_mode=:pinned_void, cerjan=cerjanFlat2D)
modelsOPTFlat2D = [rhoOPT, lambdaOPTFlat2D, muOPTFlat2D]

preparedOPT3Flat2D = if runFlatKirishimaOPT3
    pointsOPT3Flat2D = getModelPoints(modelsOPTFlat2D[1], 3,
        optRecipe2D["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT3Flat2D = (models=modelsOPTFlat2D, modelPoints=pointsOPT3Flat2D,
        Δ=recipeSpacing, modelName="kirishima_flat_OPT3_$(optFormulation)")
    numericalVolumeFlat2D = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => optRecipe2D, "modelFam" => familyOPT3Flat2D,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPTFlat2D, "representation" => "matrixfree",
    ))["numOperators"]
    preparedVolumeFlat2D = prepareLinearSystem(numericalVolumeFlat2D;
        free_surface_spacing=recipeSpacing[1:2])
    preparedClippedSurfaceFlat2D = prepare_surface_geometry_flat2D(
        clippedSurfaceRecipe2D, "kirishima_flat_surface_clipped_available",
        bcOPTFlat2D.free_surface.points)
    surfaceWholeFlat2D =
        numericalVolumeFlat2D.numericalOperators.left.geometry.freeSurfaceBoundary.points
    overlaidOPT3Flat2D = overlapBoundaryLinearSystem(
        preparedVolumeFlat2D, preparedClippedSurfaceFlat2D, surfaceWholeFlat2D;
        mode=:replace)
    @assert overlaidOPT3Flat2D.boundary_overlap_mode === :replace
    @assert !hasproperty(overlaidOPT3Flat2D, :boundary_future_operator)
    A_surfaceOPT3Flat2D = flexOPT.materializeConstantMatrix(overlaidOPT3Flat2D)
    overlapRowsOPT3Flat2D = hasproperty(overlaidOPT3Flat2D, :boundary_overlap_rows) ?
        overlaidOPT3Flat2D.boundary_overlap_rows : Int[]
    factorizationOPT3Flat2D = try
        lu(A_surfaceOPT3Flat2D)
        :regular
    catch err
        err isa SingularException ? :singular : rethrow()
    end
    auditOPT3Flat2D = (factorization=factorizationOPT3Flat2D,
        zero_boundary_rows=count(iszero,
            [norm(A_surfaceOPT3Flat2D[row, :]) for row in overlapRowsOPT3Flat2D]))
    display(auditOPT3Flat2D)
    @assert auditOPT3Flat2D.factorization === :regular
    overlaidOPT3Flat2D
else
    nothing
end

preparedOPT5Flat2D = if runFlatKirishimaOPT5
    pointsOPT5Flat2D = getModelPoints(modelsOPTFlat2D[1], 5,
        opt5Recipe2D["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT5Flat2D = (models=modelsOPTFlat2D, modelPoints=pointsOPT5Flat2D,
        Δ=recipeSpacing, modelName="kirishima_flat_OPT5_$(optFormulation)")
    numericalOPT5Flat2D = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt5Recipe2D, "modelFam" => familyOPT5Flat2D,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPTFlat2D, "representation" => "matrixfree",
    ))["numOperators"]
    preparedOPT5VolumeFlat2D = prepareLinearSystem(numericalOPT5Flat2D;
        free_surface_spacing=recipeSpacing[1:2])
    preparedClippedSurfaceOPT5Flat2D = prepare_surface_geometry_flat2D(
        clippedSurfaceRecipeOPT5_2D, "kirishima_flat_surface_clipped_available_OPT5",
        bcOPTFlat2D.free_surface.points; points_in_space=5)
    surfaceWholeOPT5Flat2D =
        numericalOPT5Flat2D.numericalOperators.left.geometry.freeSurfaceBoundary.points
    overlaidOPT5Flat2D = overlapBoundaryLinearSystem(
        preparedOPT5VolumeFlat2D, preparedClippedSurfaceOPT5Flat2D, surfaceWholeOPT5Flat2D;
        mode=:replace)
    @assert overlaidOPT5Flat2D.boundary_overlap_mode === :replace
    @assert !hasproperty(overlaidOPT5Flat2D, :boundary_future_operator)
    A_surfaceOPT5Flat2D = flexOPT.materializeConstantMatrix(overlaidOPT5Flat2D)
    overlapRowsOPT5Flat2D = hasproperty(overlaidOPT5Flat2D, :boundary_overlap_rows) ?
        overlaidOPT5Flat2D.boundary_overlap_rows : Int[]
    factorizationOPT5Flat2D = try
        lu(A_surfaceOPT5Flat2D)
        :regular
    catch err
        err isa SingularException ? :singular : rethrow()
    end
    auditOPT5Flat2D = (factorization=factorizationOPT5Flat2D,
        zero_boundary_rows=count(iszero,
            [norm(A_surfaceOPT5Flat2D[row, :]) for row in overlapRowsOPT5Flat2D]))
    display(auditOPT5Flat2D)
    @assert auditOPT5Flat2D.factorization === :regular
    overlaidOPT5Flat2D
else
    nothing
end

@show runFlatKirishimaFD3 runFlatKirishimaOPT3 runFlatKirishimaOPT5
runFlatKirishimaOPT3 && @show preparedOPT3Flat2D.spaceShape
runFlatKirishimaOPT5 && @show preparedOPT5Flat2D.spaceShape


In [ ]:
# FD3 propagation: elasticWave2D explicit stepping with a single point
# force, matching OPT3/OPT5's point-source convention in this notebook
# (no spatial Gaussian spreading, unlike the homogeneous benchmark).
if runFlatKirishimaFD3
    fdSourceIndexFlat2D = CartesianIndex(sourceIX2D, sourceIZFlat2D) +
        CartesianIndex(Tuple(preparedFD3Flat2D.padding[1, :]))
    fdCoordinatesFlat2D = elastic_wave_coordinates(
        xCoordinates2D, zCoordinates2D, preparedFD3Flat2D)
    fdStepsFlat2D = ceil(Int, simulationDuration2D / preparedFD3Flat2D.dt)
    fdOutputStrideFlat2D = max(1, round(Int, outputSampling2D / preparedFD3Flat2D.dt))
    fdFramesXFlat2D = Matrix{Float32}[copy(preparedFD3Flat2D.ux)]
    fdFramesZFlat2D = Matrix{Float32}[copy(preparedFD3Flat2D.uz)]
    fdTimesFlat2D = Float64[0.0]
    for step in 1:fdStepsFlat2D
        step_elastic_wave_2d!(preparedFD3Flat2D)
        elasticWave2D.add_ricker_source!(
            preparedFD3Flat2D, fdSourceIndexFlat2D;
            f0=sourceFrequency2D, t0=sourceDelay2D,
            amplitude=sourceForceAmplitude2D, component=:z, source_kind=:force,
        )
        if step % fdOutputStrideFlat2D == 0 || step == fdStepsFlat2D
            push!(fdFramesXFlat2D, copy(preparedFD3Flat2D.ux))
            push!(fdFramesZFlat2D, copy(preparedFD3Flat2D.uz))
            push!(fdTimesFlat2D, preparedFD3Flat2D.time)
        end
    end
    uxFDFlat2D = cat(fdFramesXFlat2D...; dims=3)
    uzFDFlat2D = cat(fdFramesZFlat2D...; dims=3)
    @show preparedFD3Flat2D.dt size(uzFDFlat2D) maximum(abs, uxFDFlat2D) maximum(abs, uzFDFlat2D)
else
    fdCoordinatesFlat2D = uxFDFlat2D = uzFDFlat2D = fdTimesFlat2D = nothing
    @warn "runFlatKirishimaFD3 = false; nothing to propagate"
end

referenceOPTFlat2D = runFlatKirishimaOPT3 ? preparedOPT3Flat2D :
    (runFlatKirishimaOPT5 ? preparedOPT5Flat2D : nothing)

if !isnothing(referenceOPTFlat2D)
    optPadding2DFlat = cerjan_padding(cerjanFlat2D)
    xOPTFlat2D = range(first(xCoordinates2D) - optPadding2DFlat[1,1]*dxOPT;
        step=dxOPT, length=referenceOPTFlat2D.spaceShape[1])
    zOPTFlat2D = range(first(zCoordinates2D) - optPadding2DFlat[1,2]*dxOPT;
        step=dxOPT, length=referenceOPTFlat2D.spaceShape[2])
    sourceIndexOPTFlat2D = CartesianIndex(
        sourceIX2D + optPadding2DFlat[1,1], sourceIZFlat2D + optPadding2DFlat[1,2])
    sourceLinearOPTFlat2D = LinearIndices(referenceOPTFlat2D.spaceShape)[sourceIndexOPTFlat2D]
    optStepsFlat2D = ceil(Int, simulationDuration2D / dtOPT)
    optOutputStrideFlat2D = max(1, round(Int, outputSampling2D / dtOPT))
    sourceDensityAtSourceFlat2D = rhoFieldOPT[sourceIX2D, sourceIZFlat2D]
    sourceScaleOPTFlat2D = sourceForceAmplitude2D * dtOPT^2 /
        (sourceDensityAtSourceFlat2D * dxOPT^2)

    function build_source_flat2D(prepared, nPastLevels)
        sourceTimes = ((1 - nPastLevels):optStepsFlat2D) .* dtOPT
        wavelet = rickerSource2D.(sourceTimes)
        source = zeros(Float64, prepared.NforcePoints, prepared.NForceField, length(sourceTimes))
        source[sourceLinearOPTFlat2D, 2, :] .= sourceScaleOPTFlat2D .* wavelet
        source
    end

    if runFlatKirishimaOPT3
        nPast3Flat2D = preparedOPT3Flat2D.timePointsUsedForOneStep - 1
        sourceOPT3Flat2D = build_source_flat2D(preparedOPT3Flat2D, nPast3Flat2D)
        propagationOPT3Flat2D = propagateLinearSystem(
            preparedOPT3Flat2D, optStepsFlat2D, dtOPT;
            sourceFull=sourceOPT3Flat2D, output_stride=optOutputStrideFlat2D,
            blowup_limit=1e8, solver_name="OPT3 clipped-available (Kirishima flat)",
            scheme=:direct,
        )
        @assert !propagationOPT3Flat2D.stopped_early
        uxOPT3Flat2D = propagationOPT3Flat2D.history[:, :, 1, :]
        uzOPT3Flat2D = propagationOPT3Flat2D.history[:, :, 2, :]
        opt3TimesFlat2D = propagationOPT3Flat2D.times
    else
        uxOPT3Flat2D = uzOPT3Flat2D = opt3TimesFlat2D = nothing
    end

    if runFlatKirishimaOPT5
        nPast5Flat2D = preparedOPT5Flat2D.timePointsUsedForOneStep - 1
        sourceOPT5Flat2D = build_source_flat2D(preparedOPT5Flat2D, nPast5Flat2D)
        propagationOPT5Flat2D = propagateLinearSystem(
            preparedOPT5Flat2D, optStepsFlat2D, dtOPT;
            sourceFull=sourceOPT5Flat2D, output_stride=optOutputStrideFlat2D,
            blowup_limit=1e8, solver_name="OPT5 clipped-available (Kirishima flat)",
            scheme=:direct,
        )
        @assert !propagationOPT5Flat2D.stopped_early
        uxOPT5Flat2D = propagationOPT5Flat2D.history[:, :, 1, :]
        uzOPT5Flat2D = propagationOPT5Flat2D.history[:, :, 2, :]
        opt5TimesFlat2D = propagationOPT5Flat2D.times
    else
        uxOPT5Flat2D = uzOPT5Flat2D = opt5TimesFlat2D = nothing
    end
else
    xOPTFlat2D = zOPTFlat2D = nothing
    uxOPT3Flat2D = uzOPT3Flat2D = opt3TimesFlat2D = nothing
    uxOPT5Flat2D = uzOPT5Flat2D = opt5TimesFlat2D = nothing
    @warn "neither runFlatKirishimaOPT3 nor runFlatKirishimaOPT5 is true; nothing to propagate"
end

@show runFlatKirishimaOPT3 runFlatKirishimaOPT5
runFlatKirishimaOPT3 && @show size(uzOPT3Flat2D) maximum(abs, uzOPT3Flat2D)
runFlatKirishimaOPT5 && @show size(uzOPT5Flat2D) maximum(abs, uzOPT5Flat2D)


In [ ]:
caseDirectoryFlat2D = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "kirishima_flat_dx$(round(Int, dxOPT))m_nrec$(length(receiverGridFlat2D))")
specfemNxElementsFlat2D = round(Int,
    (last(xCoordinates2D)-first(xCoordinates2D)) / (4dxOPT))
specfemNzElementsFlat2D = round(Int,
    (last(zCoordinates2D)-first(zCoordinates2D)) / (4dxOPT))

# Same fill-above-topography tomography trick as topography-specfem, now
# relative to the flat mask: fill the unused void samples above
# flatReferenceZ2D with the shallowest valid solid value (never sampled by
# the mesh). vpFieldOPT/vsFieldOPT/rhoFieldOPT (and materialFlat2D) are
# untouched, so FD3/OPT3/OPT5 still see the real void above the flat surface.
vpFieldSPECFEMFlat2D = copy(vpFieldOPT)
vsFieldSPECFEMFlat2D = copy(vsFieldOPT)
rhoFieldSPECFEMFlat2D = copy(rhoFieldOPT)
for ix in axes(vsFieldSPECFEMFlat2D, 1)
    valid = findall(@view materialFlat2D[ix, :])
    isempty(valid) && error("flat column $ix contains no elastic material")
    top = last(valid)
    for field in (vpFieldSPECFEMFlat2D, vsFieldSPECFEMFlat2D, rhoFieldSPECFEMFlat2D)
        field[ix, top+1:end] .= field[ix, top]
    end
end
@assert minimum(vpFieldSPECFEMFlat2D) > 0
@assert minimum(vsFieldSPECFEMFlat2D) > 0
@assert minimum(rhoFieldSPECFEMFlat2D) > 0

specfemCaseFlat2D = prepare_specfem2d_case(
    caseDirectoryFlat2D, xCoordinates2D, zCoordinates2D,
    vpFieldSPECFEMFlat2D, vsFieldSPECFEMFlat2D, rhoFieldSPECFEMFlat2D, surfaceZFlat2D;
    source=(x=epicentreX2D, z=sourcePhysicalZFlat2D),
    receiver_points=receiverGridFlat2D,
    duration=simulationDuration2D, dt=dtSPECFEM2D, f0=sourceFrequency2D,
    source_factor=sourceForceAmplitude2D, source_angle=0.0,
    source_time_function=rickerSource2D,
    nx_elements=specfemNxElementsFlat2D, nz_elements=specfemNzElementsFlat2D,
    free_surface=true, record_at_surface_same_vertical=false,
    snapshot_interval_steps=max(1, round(Int, outputSampling2D / dtSPECFEM2D)),
    snapshot_image_type=5,
    output_wavefield_dumps=false,
)
specfemRunFlat2D = if runFlatKirishimaSPECFEM2D
    run_specfem2d_case(specfemCaseFlat2D.case_directory)
elseif isfile(joinpath(specfemCaseFlat2D.case_directory, "solver.log"))
    (output=specfemCaseFlat2D.output,)
else
    nothing
end

# Same velocity -> displacement conversion as topography-specfem's own
# integrate_velocity_trace2D; redefined here since this cell runs earlier
# in notebook order (harmless duplicate `function` definition in Julia).
function integrate_velocity_trace2D(trace)
    displacement = zeros(Float64, length(trace.values))
    displacement[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .* diff(trace.time))
    (time=Float64.(trace.time), values=displacement)
end

if !isnothing(specfemRunFlat2D)
    verticalFilesFlat2D = find_specfem2d_traces(specfemRunFlat2D.output;
        component=:z, network="FX")
    horizontalFilesFlat2D = find_specfem2d_traces(specfemRunFlat2D.output;
        component=:x, network="FX")
    @assert length(verticalFilesFlat2D) == length(receiverGridFlat2D)
    @assert length(horizontalFilesFlat2D) == length(receiverGridFlat2D)
    specfemTracesZFlat2D = [read_specfem2d_trace(file;
        time_shift=specfemCaseFlat2D.time_axis_shift) for file in verticalFilesFlat2D]
    specfemTracesXFlat2D = [read_specfem2d_trace(file;
        time_shift=specfemCaseFlat2D.time_axis_shift) for file in horizontalFilesFlat2D]
    specfemWaveformsZFlat2D = integrate_velocity_trace2D.(specfemTracesZFlat2D)
    specfemWaveformsXFlat2D = integrate_velocity_trace2D.(specfemTracesXFlat2D)
    @show specfemCaseFlat2D.case_directory dtSPECFEM2D
else
    specfemWaveformsZFlat2D = specfemWaveformsXFlat2D = nothing
    @warn "runFlatKirishimaSPECFEM2D = false; set it to true above and rerun to get a reference"
end


In [ ]:
if !isnothing(specfemWaveformsZFlat2D)
    function sample_history_flat2D(history, times, xaxis, zaxis, points)
        traces = Matrix{Float64}(undef, length(times), length(points))
        for (j, point) in enumerate(points)
            ix = argmin(abs.(xaxis .- point.x))
            iz = argmin(abs.(zaxis .- point.z))
            traces[:, j] .= Float64.(history[ix, iz, :])
        end
        (time=Float64.(times), values=traces)
    end

    fd3GridXFlat2D = runFlatKirishimaFD3 ? sample_history_flat2D(
        uxFDFlat2D, fdTimesFlat2D, fdCoordinatesFlat2D.x, fdCoordinatesFlat2D.z,
        receiverGridFlat2D) : nothing
    fd3GridZFlat2D = runFlatKirishimaFD3 ? sample_history_flat2D(
        uzFDFlat2D, fdTimesFlat2D, fdCoordinatesFlat2D.x, fdCoordinatesFlat2D.z,
        receiverGridFlat2D) : nothing
    opt3GridXFlat2D = runFlatKirishimaOPT3 ? sample_history_flat2D(
        uxOPT3Flat2D, opt3TimesFlat2D, xOPTFlat2D, zOPTFlat2D, receiverGridFlat2D) : nothing
    opt3GridZFlat2D = runFlatKirishimaOPT3 ? sample_history_flat2D(
        uzOPT3Flat2D, opt3TimesFlat2D, xOPTFlat2D, zOPTFlat2D, receiverGridFlat2D) : nothing
    opt5GridXFlat2D = runFlatKirishimaOPT5 ? sample_history_flat2D(
        uxOPT5Flat2D, opt5TimesFlat2D, xOPTFlat2D, zOPTFlat2D, receiverGridFlat2D) : nothing
    opt5GridZFlat2D = runFlatKirishimaOPT5 ? sample_history_flat2D(
        uzOPT5Flat2D, opt5TimesFlat2D, xOPTFlat2D, zOPTFlat2D, receiverGridFlat2D) : nothing

    flatComparisonFigure = Figure(size=(1200, 210 * length(receiverGridFlat2D)))
    for (station, receiver) in enumerate(receiverGridFlat2D)
        for (column, (label, fd3Grid, opt3Grid, opt5Grid, specGrid)) in enumerate((
                ("x", fd3GridXFlat2D, opt3GridXFlat2D, opt5GridXFlat2D, specfemWaveformsXFlat2D),
                ("z", fd3GridZFlat2D, opt3GridZFlat2D, opt5GridZFlat2D, specfemWaveformsZFlat2D)))
            axis = Axis(flatComparisonFigure[station, column];
                xlabel=station == length(receiverGridFlat2D) ? "time (s)" : "",
                ylabel="x=$(round(receiver.x/1e3; digits=1)) km",
                title=station == 1 ? "u$label (absolute amplitude)" : "")
            !isnothing(fd3Grid) && lines!(axis, fd3Grid.time, fd3Grid.values[:, station];
                label="FD3", color=:steelblue)
            !isnothing(opt3Grid) && lines!(axis, opt3Grid.time, opt3Grid.values[:, station];
                label="OPT3", color=:darkorange)
            !isnothing(opt5Grid) && lines!(axis, opt5Grid.time, opt5Grid.values[:, station];
                label="OPT5", color=:purple, linestyle=:dash)
            lines!(axis, specGrid[station].time, specGrid[station].values;
                label="SPECFEM2D", color=:black, linewidth=2)
            station == 1 && column == 1 && axislegend(axis; position=:rt, labelsize=9)
        end
    end
    display(flatComparisonFigure)
else
    @warn "SPECFEM2D reference is missing; set runFlatKirishimaSPECFEM2D=true in flat-kirishima-config"
end
nothing


In [ ]:
# FD3: ordinary explicit finite-difference reference, same elasticWave2D
# module and call pattern as Stage 2a's flat-kirishima-operator (and
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb's fd-propagation cell),
# now driven by the real (rough) topography mask.
preparedFD32D = if runKirishimaFD3
    fdConfig2D = ElasticThreePointConfig2D(
        pointsInSpace=3, pointsInTime=3, supplementaryOrder=2, cfl=0.38,
    )
    cerjanFD2D = CerjanBoundarySpec((24, 24), (24, 0); damping=0.0053)
    bcFD2D = boundary_geometry(material2D, (dxOPT, dxOPT); cerjan=cerjanFD2D)
    modelFD2D = (ρ=seismicModel2D.ρ, Vpv=seismicModel2D.Vpv, Vsv=seismicModel2D.Vsv)
    prepare_elastic_wave_2d(
        modelFD2D, (dxOPT, dxOPT);
        material_mask=material2D,
        boundary_conditions=bcFD2D,
        config=fdConfig2D,
    )
else
    nothing
end
runKirishimaFD3 && @show preparedFD32D.dt

# OPT3/OPT5 weak recipes. Hoisted out of any run-flag guard (unlike the
# original OPT3-only version of this cell) so that OPT5's recipe can be
# built independently of runKirishimaOPT3, and so the disk cache stays
# shared with flat-kirishima-operator and
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb's own recipes -- only the
# material fields differ between those three notebooks.
optParameters2D = Dict{String,Any}(
    "famousEquationType" => "2DsismoTimeIsoHeteroSingleForce",
    "Δ" => recipeSpacing,
    "orderBtime" => 1, "orderBspace" => 1,
    "pointsInSpace" => 3, "pointsInTime" => 3,
    "supplementaryOrder" => 2,
    "variationalForm" => DEFAULT_FORM[],
    "taylor_inverse_mode" => :weak_operator_optimized,
    "fieldItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "materItpl" => (ptsSpace=1, ptsTime=1, offsetSpace=1.0,
        offsetTime=1, YorderBspace=-1, YorderBtime=-1),
    "recipe_backend" => backend,
)
# Same 3x2 clipped-available surface recipe as
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb, audited by
# scripts/audit_opt3_weak_cross_partials.jl.
clippedSurfaceParameters2D = copy(optParameters2D)
clippedSurfaceParameters2D["pointsInSpace"] = (3, 2)
clippedSurfaceParameters2D["nuCentre"] = (2, 2)
clippedSurfaceParameters2D["fieldItpl"] = (
    ptsSpace=(3, 2), ptsTime=1, offsetSpace=(0.0, 0.0),
    offsetTime=1, YorderBspace=1, YorderBtime=-1,
)
clippedSurfaceParameters2D["trialFunctionRefPoints"] = (
    (1, 2, 3), (1, 2, 3), (1, 2, 3),
)
clippedSurfaceParameters2D["testIntegrationBounds"] = (
    (1.0, 3.0), (1.0, 2.0), (1.0, 3.0),
)
clippedSurfaceParameters2D["exactTaylorTotalDegree"] = 2

# Five-point OPT5 recipe and its clipped_available free-surface closure --
# the same 5->3 generalisation validated in
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb's opt-recipes/opt-operator
# cells and reused unchanged in flat-kirishima-operator above.
opt5Parameters2D = copy(optParameters2D)
opt5Parameters2D["orderBspace"] = 1
opt5Parameters2D["pointsInSpace"] = 5
opt5Parameters2D["fieldItpl"] = merge(optParameters2D["fieldItpl"], (offsetSpace=2.0,))
opt5Parameters2D["materItpl"] = merge(optParameters2D["materItpl"], (offsetSpace=2.0,))
clippedSurfaceParametersOPT5_2D = copy(opt5Parameters2D)
clippedSurfaceParametersOPT5_2D["pointsInSpace"] = (5, 3)
clippedSurfaceParametersOPT5_2D["nuCentre"] = (3, 3)
clippedSurfaceParametersOPT5_2D["fieldItpl"] = (
    ptsSpace=(5, 3), ptsTime=1, offsetSpace=(0.0, 0.0),
    offsetTime=1, YorderBspace=1, YorderBtime=-1,
)
clippedSurfaceParametersOPT5_2D["trialFunctionRefPoints"] = (
    (1, 2, 3, 4, 5), (1, 2, 3, 4, 5), (1, 2, 3),
)
clippedSurfaceParametersOPT5_2D["testIntegrationBounds"] = (
    (1.0, 5.0), (1.0, 3.0), (1.0, 3.0),
)
clippedSurfaceParametersOPT5_2D["exactTaylorTotalDegree"] = 4

# The weak-form basis only depends on the stencil geometry, not on
# this model's fields, so this cache is shared with
# HomogeneousElastic2DBenchmark_FreeSurface.ipynb's own recipes.
recipeCacheVersion2D = 9
function cachedOPTRecipe2D(parameters, prefix)
    cacheParameters = Dict{String,Any}(
        key => value for (key, value) in parameters if key != "recipe_backend")
    cacheParameters["recipe_cache_version"] = recipeCacheVersion2D
    function produceRecipe(config)
        runtimeParameters = Dict{String,Any}(config)
        pop!(runtimeParameters, "hash_id", nothing)
        pop!(runtimeParameters, "recipe_cache_version", nothing)
        runtimeParameters["recipe_backend"] = backend
        makeOPTsemiSymbolic(runtimeParameters)
    end
    myProduceOrLoad(produceRecipe, cacheParameters, "semiSymbolic", prefix)
end
optRecipe2D = cachedOPTRecipe2D(optParameters2D, "elastic2D_OPT3")
clippedSurfaceRecipe2D = cachedOPTRecipe2D(
    clippedSurfaceParameters2D, "elastic2D_OPT3_surface_clipped_available")
if runKirishimaOPT5
    coefficientGatePath2D = joinpath(flexopt_root, "data",
        "elastic_lhs_coefficient_gate.jld2")
    isfile(coefficientGatePath2D) || error(
        "Run ElasticLHSOperatorAudit.ipynb before OPT5 propagation")
    coefficientGate2D = load(coefficientGatePath2D)
    @assert all(result.passed for result in values(coefficientGate2D["coefficient_health"]))
end
opt5Recipe2D = runKirishimaOPT5 ?
    cachedOPTRecipe2D(opt5Parameters2D, "elastic2D_OPT5") : nothing
clippedSurfaceRecipeOPT5_2D = runKirishimaOPT5 ?
    cachedOPTRecipe2D(clippedSurfaceParametersOPT5_2D,
        "elastic2D_OPT5_surface_clipped_available") : nothing

cerjan2D = CerjanBoundarySpec((24, 24), (24, 0); damping=0.0053)
bcOPT2D = boundary_geometry(material2D, (dxOPT, dxOPT);
    free_surface_mode=:pinned_void, cerjan=cerjan2D)
modelsOPT2D = [rhoOPT, lambdaOPT, muOPT]

function prepare_surface_geometry2D(recipe, name; points_in_space=3)
    surfacePoints = getModelPoints(modelsOPT2D[1], points_in_space,
        recipe["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    surfaceFamily = (models=modelsOPT2D, modelPoints=surfacePoints,
        Δ=recipeSpacing, modelName=name)
    numerical = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => recipe, "modelFam" => surfaceFamily,
        # Same Cerjan padding as the volume operator, given directly
        # (not through "boundaryConditions", which would also attach a
        # second freeSurfaceBoundary on top of maskedRegionInSpace
        # below). Without this, this surface-only grid comes out
        # unpadded while numericalVolume2D is padded, and
        # overlapBoundaryLinearSystem rejects the size mismatch
        # (DimensionMismatch: volume and boundary grids differ) — the
        # flat benchmark never hits this because it runs with no
        # Cerjan layer at all (applyCerjan=false there).
        "absorbingBoundaries" => cerjan_padding(cerjan2D),
        "maskedRegionInSpace" => bcOPT2D.free_surface.points,
        "boundaryConditions" => nothing,
        "representation" => "matrixfree",
    ))["numOperators"]
    prepareLinearSystem(numerical)
end

preparedOPT2D = if runKirishimaOPT3
    pointsOPT2D = getModelPoints(modelsOPT2D[1], 3,
        optRecipe2D["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT2D = (models=modelsOPT2D, modelPoints=pointsOPT2D,
        Δ=recipeSpacing, modelName="kirishima_topography_OPT3_$(optFormulation)")
    numericalVolume2D = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => optRecipe2D, "modelFam" => familyOPT2D,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPT2D, "representation" => "matrixfree",
    ))["numOperators"]
    preparedVolume2D = prepareLinearSystem(numericalVolume2D;
        free_surface_spacing=recipeSpacing[1:2])

    preparedClippedSurface2D = prepare_surface_geometry2D(
        clippedSurfaceRecipe2D, "kirishima_topography_surface_clipped_available")
    surfaceWhole2D =
        numericalVolume2D.numericalOperators.left.geometry.freeSurfaceBoundary.points
    overlaid2D = overlapBoundaryLinearSystem(
        preparedVolume2D, preparedClippedSurface2D, surfaceWhole2D; mode=:replace)
    @assert overlaid2D.boundary_overlap_mode === :replace
    @assert !hasproperty(overlaid2D, :boundary_future_operator)

    A_surface2D = flexOPT.materializeConstantMatrix(overlaid2D)
    boundaryOverlapRows2D = hasproperty(overlaid2D, :boundary_overlap_rows) ?
        overlaid2D.boundary_overlap_rows : Int[]
    surfaceRowNorms2D = [norm(A_surface2D[row, :]) for row in boundaryOverlapRows2D]
    surfaceFactorization2D = try
        lu(A_surface2D)
        :regular
    catch err
        err isa SingularException ? :singular : rethrow()
    end
    @assert surfaceFactorization2D === :regular

    # Unlike the flat benchmark, the surface normals are not expected to
    # be vertical here: check that they are unit vectors pointing away
    # from the material instead, and report their actual range.
    surfaceNormals2D = bcOPT2D.free_surface.normals
    @assert all(n -> isapprox(hypot(n...), 1.0; atol=1e-6), surfaceNormals2D)
    @assert all(n -> n[2] > 0.0, surfaceNormals2D) "a surface normal points into the ground; check the material mask"
    freeSurfaceAudit2D = (
        surface_points=length(bcOPT2D.free_surface.points),
        overlap_rows=length(boundaryOverlapRows2D),
        zero_boundary_rows=count(iszero, surfaceRowNorms2D),
        normal_x_extrema=extrema(first.(surfaceNormals2D)),
        normal_z_extrema=extrema(last.(surfaceNormals2D)),
        surface_closure=:clipped_available,
    )
    display(freeSurfaceAudit2D)
    overlaid2D
else
    @warn "runKirishimaOPT3 = false; set it to true above and rerun to assemble OPT3"
    nothing
end

# Same natural-weak free-surface treatment for OPT5 as flat-kirishima-operator
# above, now on the real topography mask/normals.
preparedOPT5_2D = if runKirishimaOPT5
    pointsOPT5_2D = getModelPoints(modelsOPT2D[1], 5,
        opt5Recipe2D["recette"].numbersOfTheSystem.numbersOfTheSystemL.timeMarching)
    familyOPT5_2D = (models=modelsOPT2D, modelPoints=pointsOPT5_2D,
        Δ=recipeSpacing, modelName="kirishima_topography_OPT5_$(optFormulation)")
    numericalOPT5_2D = numericalOperatorConstruction(Dict{String,Any}(
        "optRec" => opt5Recipe2D, "modelFam" => familyOPT5_2D,
        "absorbingBoundaries" => nothing, "maskedRegionInSpace" => nothing,
        "boundaryConditions" => bcOPT2D, "representation" => "matrixfree",
    ))["numOperators"]
    preparedOPT5Volume2D = prepareLinearSystem(numericalOPT5_2D;
        free_surface_spacing=recipeSpacing[1:2])
    preparedClippedSurfaceOPT5_2D = prepare_surface_geometry2D(
        clippedSurfaceRecipeOPT5_2D, "kirishima_topography_surface_clipped_available_OPT5";
        points_in_space=5)
    surfaceWholeOPT5_2D =
        numericalOPT5_2D.numericalOperators.left.geometry.freeSurfaceBoundary.points
    overlaidOPT5_2D = overlapBoundaryLinearSystem(
        preparedOPT5Volume2D, preparedClippedSurfaceOPT5_2D, surfaceWholeOPT5_2D;
        mode=:replace)
    @assert overlaidOPT5_2D.boundary_overlap_mode === :replace
    @assert !hasproperty(overlaidOPT5_2D, :boundary_future_operator)
    A_surfaceOPT5_2D = flexOPT.materializeConstantMatrix(overlaidOPT5_2D)
    overlapRowsOPT5_2D = hasproperty(overlaidOPT5_2D, :boundary_overlap_rows) ?
        overlaidOPT5_2D.boundary_overlap_rows : Int[]
    factorizationOPT5_2D = try
        lu(A_surfaceOPT5_2D)
        :regular
    catch err
        err isa SingularException ? :singular : rethrow()
    end
    auditOPT5_2D = (factorization=factorizationOPT5_2D,
        zero_boundary_rows=count(iszero,
            [norm(A_surfaceOPT5_2D[row, :]) for row in overlapRowsOPT5_2D]))
    display(auditOPT5_2D)
    @assert auditOPT5_2D.factorization === :regular
    overlaidOPT5_2D
else
    nothing
end

@show runKirishimaFD3 runKirishimaOPT3 runKirishimaOPT5
runKirishimaOPT5 && @show preparedOPT5_2D.spaceShape


In [ ]:
# Hoisted so FD3/OPT5 propagation below can use them even when
# runKirishimaOPT3=false (they depend only on config, not on preparedOPT2D).
optSteps2D = ceil(Int, simulationDuration2D / dtOPT)
optOutputStride2D = max(1, round(Int, outputSampling2D / dtOPT))
if runKirishimaOPT3
    optPadding2D = cerjan_padding(bcOPT2D.cerjan)
    xOPT2D = range(first(xCoordinates2D) - optPadding2D[1,1]*dxOPT;
        step=dxOPT, length=preparedOPT2D.spaceShape[1])
    zOPT2D = range(first(zCoordinates2D) - optPadding2D[1,2]*dxOPT;
        step=dxOPT, length=preparedOPT2D.spaceShape[2])
    sourceIndexOPT2D = CartesianIndex(
        sourceIX2D + optPadding2D[1,1], sourceIZ2D + optPadding2D[1,2])
    sourceLinearOPT2D = LinearIndices(preparedOPT2D.spaceShape)[sourceIndexOPT2D]

    optSteps2D = ceil(Int, simulationDuration2D / dtOPT)
    optOutputStride2D = max(1, round(Int, outputSampling2D / dtOPT))
    nPastForceLevels2D = preparedOPT2D.timePointsUsedForOneStep - 1
    sourceTimesOPT2D = ((1 - nPastForceLevels2D):optSteps2D) .* dtOPT
    waveletOPT2D = rickerSource2D.(sourceTimesOPT2D)
    # dt^2 F / (ρ Δx Δz): same force-to-displacement conversion as
    # elasticWave2D.add_ricker_source!(...; source_kind=:force).
    sourceDensityAtSource2D = rhoFieldOPT[sourceIX2D, sourceIZ2D]
    sourceScaleOPT2D = sourceForceAmplitude2D * dtOPT^2 /
        (sourceDensityAtSource2D * dxOPT^2)
    sourceOPT2D = zeros(Float64, preparedOPT2D.NforcePoints,
        preparedOPT2D.NForceField, length(sourceTimesOPT2D))
    sourceOPT2D[sourceLinearOPT2D, 2, :] .= sourceScaleOPT2D .* waveletOPT2D

    propagationOPT2D = propagateLinearSystem(
        preparedOPT2D, optSteps2D, dtOPT;
        sourceFull=sourceOPT2D, output_stride=optOutputStride2D,
        blowup_limit=1e8, solver_name="OPT3 clipped-available (Kirishima topography)",
        scheme=:direct,
    )
    @assert !propagationOPT2D.stopped_early
    uxOPT2D = propagationOPT2D.history[:, :, 1, :]
    uzOPT2D = propagationOPT2D.history[:, :, 2, :]
    optTimes2D = propagationOPT2D.times
    @show dtOPT size(uzOPT2D) maximum(abs, uxOPT2D) maximum(abs, uzOPT2D)
    @show propagationOPT2D.timing
else
    xOPT2D = zOPT2D = optTimes2D = uxOPT2D = uzOPT2D = nothing
    @warn "runKirishimaOPT3 = false; nothing to propagate"
end

# FD3 propagation: elasticWave2D explicit stepping with a single point
# force, matching OPT3/OPT5's point-source convention in this notebook
# (same pattern as flat-kirishima-propagation above).
if runKirishimaFD3
    fdSourceIndex2D = CartesianIndex(sourceIX2D, sourceIZ2D) +
        CartesianIndex(Tuple(preparedFD32D.padding[1, :]))
    fdCoordinates2D = elastic_wave_coordinates(
        xCoordinates2D, zCoordinates2D, preparedFD32D)
    fdSteps2D = ceil(Int, simulationDuration2D / preparedFD32D.dt)
    fdOutputStride2D = max(1, round(Int, outputSampling2D / preparedFD32D.dt))
    fdFramesX2D = Matrix{Float32}[copy(preparedFD32D.ux)]
    fdFramesZ2D = Matrix{Float32}[copy(preparedFD32D.uz)]
    fdTimes2D = Float64[0.0]
    for step in 1:fdSteps2D
        step_elastic_wave_2d!(preparedFD32D)
        elasticWave2D.add_ricker_source!(
            preparedFD32D, fdSourceIndex2D;
            f0=sourceFrequency2D, t0=sourceDelay2D,
            amplitude=sourceForceAmplitude2D, component=:z, source_kind=:force,
        )
        if step % fdOutputStride2D == 0 || step == fdSteps2D
            push!(fdFramesX2D, copy(preparedFD32D.ux))
            push!(fdFramesZ2D, copy(preparedFD32D.uz))
            push!(fdTimes2D, preparedFD32D.time)
        end
    end
    uxFD2D = cat(fdFramesX2D...; dims=3)
    uzFD2D = cat(fdFramesZ2D...; dims=3)
    @show preparedFD32D.dt size(uzFD2D) maximum(abs, uxFD2D) maximum(abs, uzFD2D)
else
    fdCoordinates2D = uxFD2D = uzFD2D = fdTimes2D = nothing
    @warn "runKirishimaFD3 = false; nothing to propagate"
end

# OPT5 propagation: same point-source and free-surface-clipped operator
# pattern as OPT3 above, built independently of runKirishimaOPT3 so it
# works even when only OPT5 is requested.
if runKirishimaOPT5
    optPadding5_2D = cerjan_padding(bcOPT2D.cerjan)
    xOPT5_2D = range(first(xCoordinates2D) - optPadding5_2D[1,1]*dxOPT;
        step=dxOPT, length=preparedOPT5_2D.spaceShape[1])
    zOPT5_2D = range(first(zCoordinates2D) - optPadding5_2D[1,2]*dxOPT;
        step=dxOPT, length=preparedOPT5_2D.spaceShape[2])
    sourceIndexOPT5_2D = CartesianIndex(
        sourceIX2D + optPadding5_2D[1,1], sourceIZ2D + optPadding5_2D[1,2])
    sourceLinearOPT5_2D = LinearIndices(preparedOPT5_2D.spaceShape)[sourceIndexOPT5_2D]

    nPastForceLevelsOPT5_2D = preparedOPT5_2D.timePointsUsedForOneStep - 1
    sourceTimesOPT5_2D = ((1 - nPastForceLevelsOPT5_2D):optSteps2D) .* dtOPT
    waveletOPT5_2D = rickerSource2D.(sourceTimesOPT5_2D)
    sourceDensityAtSourceOPT5_2D = rhoFieldOPT[sourceIX2D, sourceIZ2D]
    sourceScaleOPT5_2D = sourceForceAmplitude2D * dtOPT^2 /
        (sourceDensityAtSourceOPT5_2D * dxOPT^2)
    sourceOPT5_2D = zeros(Float64, preparedOPT5_2D.NforcePoints,
        preparedOPT5_2D.NForceField, length(sourceTimesOPT5_2D))
    sourceOPT5_2D[sourceLinearOPT5_2D, 2, :] .= sourceScaleOPT5_2D .* waveletOPT5_2D

    propagationOPT5_2D = propagateLinearSystem(
        preparedOPT5_2D, optSteps2D, dtOPT;
        sourceFull=sourceOPT5_2D, output_stride=optOutputStride2D,
        blowup_limit=1e8, solver_name="OPT5 clipped-available (Kirishima topography)",
        scheme=:direct,
    )
    @assert !propagationOPT5_2D.stopped_early
    uxOPT5_2D = propagationOPT5_2D.history[:, :, 1, :]
    uzOPT5_2D = propagationOPT5_2D.history[:, :, 2, :]
    opt5Times2D = propagationOPT5_2D.times
    @show size(uzOPT5_2D) maximum(abs, uxOPT5_2D) maximum(abs, uzOPT5_2D)
else
    xOPT5_2D = zOPT5_2D = opt5Times2D = uxOPT5_2D = uzOPT5_2D = nothing
    @warn "runKirishimaOPT5 = false; nothing to propagate"
end


In [ ]:
caseDirectory2D = joinpath(flexopt_root, "data", "specfem2d_benchmarks",
    "kirishima_topography_dx$(round(Int, dxOPT))m_nrec$(length(receiverGrid2D))")
specfemNxElements2D = round(Int,
    (last(xCoordinates2D)-first(xCoordinates2D)) / (4dxOPT))
specfemNzElements2D = round(Int,
    (last(zCoordinates2D)-first(zCoordinates2D)) / (4dxOPT))

# SPECFEM's tomography file is rectangular although the spectral mesh
# ends at the free surface: fill the unused samples above the topographic
# interface with the shallowest valid solid value (never sampled by the
# mesh). Same fix as KirishimaElastic2DBenchmark.ipynb's prepare-tomography
# cell; vpFieldOPT/vsFieldOPT/rhoFieldOPT (and material2D) are untouched,
# so OPT3 still sees the real void above the topography.
vpFieldSPECFEM2D = copy(vpFieldOPT)
vsFieldSPECFEM2D = copy(vsFieldOPT)
rhoFieldSPECFEM2D = copy(rhoFieldOPT)
for ix in axes(vsFieldSPECFEM2D, 1)
    valid = findall((@view vsFieldSPECFEM2D[ix, :]) .> 0)
    isempty(valid) && error("column $ix contains no elastic material")
    top = last(valid)
    for field in (vpFieldSPECFEM2D, vsFieldSPECFEM2D, rhoFieldSPECFEM2D)
        field[ix, top+1:end] .= field[ix, top]
    end
end
@assert minimum(vpFieldSPECFEM2D) > 0
@assert minimum(vsFieldSPECFEM2D) > 0
@assert minimum(rhoFieldSPECFEM2D) > 0

specfemCase2D = prepare_specfem2d_case(
    caseDirectory2D, xCoordinates2D, zCoordinates2D,
    vpFieldSPECFEM2D, vsFieldSPECFEM2D, rhoFieldSPECFEM2D, surfaceZ2D;
    source=(x=epicentreX2D, z=sourcePhysicalZ2D),
    receiver_points=receiverGrid2D,
    duration=simulationDuration2D, dt=dtSPECFEM2D, f0=sourceFrequency2D,
    source_factor=sourceForceAmplitude2D, source_angle=0.0,
    source_time_function=rickerSource2D,
    nx_elements=specfemNxElements2D, nz_elements=specfemNzElements2D,
    free_surface=true, record_at_surface_same_vertical=false,
    snapshot_interval_steps=max(1, round(Int, outputSampling2D / dtSPECFEM2D)),
    snapshot_image_type=5, # vertical velocity JPEG snapshots -> the video below
    # read_specfem2d_wavefield_dumps assumes a rectangular tensor grid and
    # cannot parse a topography-following mesh ("wavefield dump is not a
    # complete tensor grid"), so the binary dumps are not requested here;
    # topography-video uses SPECFEM's own JPEG snapshots instead.
    output_wavefield_dumps=false,
)
specfemRun2D = if runKirishimaSPECFEM2D
    run_specfem2d_case(specfemCase2D.case_directory)
elseif isfile(joinpath(specfemCase2D.case_directory, "solver.log"))
    (output=specfemCase2D.output,)
else
    nothing
end

function integrate_velocity_trace2D(trace)
    # SPECFEM2D's seismotype=2 traces are velocity; OPT3's propagated
    # history is displacement (same conversion as
    # HomogeneousElastic2DBenchmark_FreeSurface.ipynb's specfem cell).
    displacement = zeros(Float64, length(trace.values))
    displacement[2:end] .= cumsum(
        ((trace.values[1:end-1] .+ trace.values[2:end]) ./ 2) .* diff(trace.time))
    (time=Float64.(trace.time), values=displacement)
end

if !isnothing(specfemRun2D)
    verticalFiles2D = find_specfem2d_traces(specfemRun2D.output;
        component=:z, network="FX")
    horizontalFiles2D = find_specfem2d_traces(specfemRun2D.output;
        component=:x, network="FX")
    @assert length(verticalFiles2D) == length(receiverGrid2D)
    @assert length(horizontalFiles2D) == length(receiverGrid2D)
    specfemTracesZ2D = [read_specfem2d_trace(file;
        time_shift=specfemCase2D.time_axis_shift) for file in verticalFiles2D]
    specfemTracesX2D = [read_specfem2d_trace(file;
        time_shift=specfemCase2D.time_axis_shift) for file in horizontalFiles2D]
    specfemWaveformsZ2D = integrate_velocity_trace2D.(specfemTracesZ2D)
    specfemWaveformsX2D = integrate_velocity_trace2D.(specfemTracesX2D)
    @show specfemCase2D.case_directory dtSPECFEM2D
else
    specfemWaveformsZ2D = specfemWaveformsX2D = nothing
    @warn "runKirishimaSPECFEM2D = false; set it to true above and rerun to get a reference"
end

In [ ]:
if !isnothing(specfemWaveformsZ2D)
    function sample_history2D(history, times, xaxis, zaxis, points)
        traces = Matrix{Float64}(undef, length(times), length(points))
        for (j, point) in enumerate(points)
            ix = argmin(abs.(xaxis .- point.x))
            iz = argmin(abs.(zaxis .- point.z))
            traces[:, j] .= Float64.(history[ix, iz, :])
        end
        (time=Float64.(times), values=traces)
    end

    fd3GridX2D = !isnothing(uxFD2D) ? sample_history2D(
        uxFD2D, fdTimes2D, fdCoordinates2D.x, fdCoordinates2D.z, receiverGrid2D) : nothing
    fd3GridZ2D = !isnothing(uzFD2D) ? sample_history2D(
        uzFD2D, fdTimes2D, fdCoordinates2D.x, fdCoordinates2D.z, receiverGrid2D) : nothing
    optGridX2D = !isnothing(uxOPT2D) ?
        sample_history2D(uxOPT2D, optTimes2D, xOPT2D, zOPT2D, receiverGrid2D) : nothing
    optGridZ2D = !isnothing(uzOPT2D) ?
        sample_history2D(uzOPT2D, optTimes2D, xOPT2D, zOPT2D, receiverGrid2D) : nothing
    opt5GridX2D = !isnothing(uxOPT5_2D) ?
        sample_history2D(uxOPT5_2D, opt5Times2D, xOPT5_2D, zOPT5_2D, receiverGrid2D) : nothing
    opt5GridZ2D = !isnothing(uzOPT5_2D) ?
        sample_history2D(uzOPT5_2D, opt5Times2D, xOPT5_2D, zOPT5_2D, receiverGrid2D) : nothing

    stationMetrics2D = map(eachindex(receiverGrid2D)) do station
        metrics = (x_km=receiverGrid2D[station].x/1e3,)
        if !isnothing(optGridZ2D)
            candidateZ = (time=optGridZ2D.time, values=optGridZ2D.values[:, station])
            candidateX = (time=optGridX2D.time, values=optGridX2D.values[:, station])
            metrics = merge(metrics, (
                uz_OPT3=waveform_metrics(specfemWaveformsZ2D[station], candidateZ; samples=1001),
                ux_OPT3=waveform_metrics(specfemWaveformsX2D[station], candidateX; samples=1001)))
        end
        if !isnothing(opt5GridZ2D)
            candidateZ5 = (time=opt5GridZ2D.time, values=opt5GridZ2D.values[:, station])
            candidateX5 = (time=opt5GridX2D.time, values=opt5GridX2D.values[:, station])
            metrics = merge(metrics, (
                uz_OPT5=waveform_metrics(specfemWaveformsZ2D[station], candidateZ5; samples=1001),
                ux_OPT5=waveform_metrics(specfemWaveformsX2D[station], candidateX5; samples=1001)))
        end
        metrics
    end
    foreach(display, stationMetrics2D)

    waveformFitFigure2D = Figure(size=(1200, 210 * length(receiverGrid2D)))
    for (station, receiver) in enumerate(receiverGrid2D)
        for (column, (label, fd3Grid, optGrid, opt5Grid, specGrid)) in enumerate((
                ("x", fd3GridX2D, optGridX2D, opt5GridX2D, specfemWaveformsX2D),
                ("z", fd3GridZ2D, optGridZ2D, opt5GridZ2D, specfemWaveformsZ2D)))
            axis = Axis(waveformFitFigure2D[station, column];
                xlabel=station == length(receiverGrid2D) ? "time (s)" : "",
                ylabel="x=$(round(receiver.x/1e3; digits=1)) km",
                title=station == 1 ? "u$label (absolute amplitude)" : "")
            !isnothing(fd3Grid) && lines!(axis, fd3Grid.time, fd3Grid.values[:, station];
                label="FD3", color=:steelblue)
            !isnothing(optGrid) && lines!(axis, optGrid.time, optGrid.values[:, station];
                label="OPT3", color=:darkorange)
            !isnothing(opt5Grid) && lines!(axis, opt5Grid.time, opt5Grid.values[:, station];
                label="OPT5", color=:purple, linestyle=:dash)
            lines!(axis, specGrid[station].time, specGrid[station].values;
                label="SPECFEM2D", color=:black, linewidth=2)
            station == 1 && column == 1 && axislegend(axis; position=:rt, labelsize=9)
        end
    end
    display(waveformFitFigure2D)
else
    @warn "SPECFEM2D results are missing; set runKirishimaSPECFEM2D=true in topography-opt-config"
end
nothing


In [ ]:
# read_specfem2d_wavefield_dumps cannot parse a topography-following mesh
# (see topography-specfem), so this is OPT3's own heatmap video and
# SPECFEM2D's own JPEG-snapshot video (make_specfem2d_snapshot_video,
# independent of output_wavefield_dumps) side by side as two files,
# rather than a single merged per-frame comparison.
videoDirectory2D = joinpath(flexopt_root, "data", "kirishimaElastic2DTopography")
mkpath(videoDirectory2D)

if !isnothing(uxOPT2D)
    function record_kirishima_opt3(filepath; framerate=15)
        commonPeak = max(maximum(abs, uzOPT2D), eps(Float64))
        colorRange = (-commonPeak, commonPeak)
        currentTime = Observable(first(optTimes2D))
        frame = Observable(Float32.(uzOPT2D[:, :, 1]))
        figure = Figure(size=(760, 560))
        axis = Axis(figure[1, 1]; xlabel="x (km)", ylabel="z (km)",
            title=@lift("OPT3 clipped-available — t = $(round($currentTime; digits=2)) s"),
            aspect=DataAspect())
        plot = heatmap!(axis, collect(xOPT2D)./1e3, collect(zOPT2D)./1e3,
            frame; colormap=:balance, colorrange=colorRange)
        lines!(axis, xCoordinates2D./1e3, surfaceZ2D./1e3; color=:black, linewidth=1)
        scatter!(axis, [epicentreX2D/1e3], [sourcePhysicalZ2D/1e3];
            marker=:star5, color=:gold, strokecolor=:black, markersize=16)
        Colorbar(figure[1, 2], plot; label="u_z (m)")
        record(figure, filepath, eachindex(optTimes2D); framerate=framerate) do frameIndex
            currentTime[] = optTimes2D[frameIndex]
            frame[] = Float32.(uzOPT2D[:, :, frameIndex])
        end
        filepath
    end
    opt3Video2D = record_kirishima_opt3(
        joinpath(videoDirectory2D, "OPT3_topography.mp4"))
    @show opt3Video2D
else
    opt3Video2D = nothing
    @warn "runKirishimaOPT3 = false; nothing to animate"
end

if !isnothing(specfemRun2D)
    specfemVideo2D = make_specfem2d_snapshot_video(specfemRun2D.output;
        output_path=joinpath(videoDirectory2D, "SPECFEM2D_topography.mp4"),
        framerate=15,
    )
    @show specfemVideo2D.path
else
    specfemVideo2D = nothing
    @warn "runKirishimaSPECFEM2D = false; set it to true above and rerun to get a video"
end
nothing